# llm-d: Kubernetes-Native Distributed LLM Inference

**A theory-only reference notebook** on `llm-d` — what it is, why it exists, how its
architecture works, and how it fits into the broader LLM-serving ecosystem
(vLLM, TGI, KServe, Ray Serve, Triton, etc).

> No executable code cells — this project is deployed via Helm charts and Kubernetes
> YAML manifests, not a Python library, so this notebook is a conceptual + reference guide.

### Table of Contents
1. [What Problem Does llm-d Solve?](#1)
2. [Project Facts: Origin, Governance & Backers](#2)
3. [Core Architecture](#3)
4. [Feature A — Prefill/Decode (P/D) Disaggregation](#4)
5. [Feature B — Native Multi-Node Orchestration (LeaderWorkerSet)](#5)
6. [How llm-d Fits the Broader Ecosystem](#6)
7. [Comparison: llm-d vs. Other Serving Layers](#7)
8. [Cheat Sheet & Key Terms](#8)
9. [Getting Started](#9)


## 1. What Problem Does llm-d Solve? <a id="1"></a>

Kubernetes is the industry-standard orchestrator for stateless web workloads (a checkout
page, a REST API) — it scales pods, restarts crashed containers, and load-balances traffic
round-robin. But **it has no concept of what happens inside an LLM inference request**:

- It doesn't know what a **token** or a **KV cache** is.
- It doesn't know that two requests hitting the *same* pod can reuse cached attention
  state, while two requests hitting *different* pods each pay the full compute cost again.
- It doesn't know that a single "generate a response" request actually has **two very
  different compute profiles** inside it (prefill vs. decode — see [Section 4](#4)).
- It doesn't know that some models don't fit on one GPU and need multiple GPUs acting as
  one logical inference engine (see [Section 5](#5)).

**llm-d** is a Kubernetes-native, open-source framework that teaches Kubernetes these
concepts. It sits as an intelligence layer *on top of* Kubernetes — not a replacement for
it — turning a generic container scheduler into an inference-aware serving platform.

Rather than every company hand-rolling this logic (custom routing sidecars, custom
autoscalers keyed on GPU memory, custom scripts to wire up multi-GPU tensor parallelism),
llm-d standardizes it as a shared, vendor-neutral piece of cloud-native infrastructure.


## 2. Project Facts: Origin, Governance & Backers <a id="2"></a>

A few points in the original brief were slightly off or incomplete — corrected/expanded
below:

| Fact | Detail |
|---|---|
| **Origin** | Created by **Neural Magic**, later acquired by **Red Hat** (2025) |
| **Public launch** | May 2025 |
| **Founding collaborators** | Red Hat, Google Cloud, IBM Research, **NVIDIA**, and **CoreWeave** — CoreWeave is a founding backer that's easy to miss |
| **Later joined as contributors** | AMD, Cisco, Hugging Face, Intel, Lambda, Mistral AI — plus academic support from UC Berkeley and the University of Chicago |
| **Governance** | Accepted into the **CNCF Sandbox** on **March 24, 2026**, announced at KubeCon Europe 2026 in Amsterdam — i.e. it is *hosted by* the CNCF, not merely "backed by" a fixed set of four companies |
| **Built on top of** | **vLLM** (the inference engine most InferencePools run) and the **Kubernetes Gateway API Inference Extension** (the upstream spec the Inference Gateway implements) |
| **Design philosophy** | "any model, any accelerator, any cloud" — explicit hardware neutrality across NVIDIA, AMD, Intel, and Google TPUs |

**Why this matters:** llm-d is not a single-vendor product wrapped in an open-source
license — it's a community-governed CNCF project explicitly built as a shared, neutral
standard, the same way Kubernetes itself is. That's a meaningfully different trust and
longevity story than a vendor tool.


## 3. Core Architecture <a id="3"></a>

Installing llm-d (typically via a **Helm chart**) deploys a set of controllers that give
Kubernetes new, inference-aware behaviors.

```
                  [ Incoming User Prompts ]
                             │
                             ▼
              ┌─────────────────────────────┐
              │    Inference Gateway (IGW)   │
              │      (Cache-Aware Routing)   │
              └──────────────┬──────────────┘
                              │
         ┌────────────────────┴────────────────────┐
         ▼                                          ▼
┌──────────────────┐                       ┌──────────────────┐
│  InferencePool    │                       │  InferencePool    │
│  (Model: Qwen)     │                       │  (Model: Llama)    │
│  ┌────────────┐    │                       │  ┌────────────┐    │
│  │ Pod (vLLM)  │    │                       │  │ Pod (vLLM)  │    │
│  └────────────┘    │                       │  └────────────┘    │
└──────────────────┘                       └──────────────────┘
```

### A. Custom Resource Definitions (CRDs)

llm-d extends the Kubernetes API with new object types you declare in standard YAML:

- **`InferencePool`** — instead of deploying a raw `Deployment` of pods, you declare a
  pool: which model to run (e.g. `Qwen2.5-7B`), which engine backs it (**vLLM** is the
  reference/primary engine; TGI and Triton are also supported), and scale bounds. llm-d's
  controllers reconcile the underlying GPU workers to match.
- **`InferenceModel`** — tracks a specific model configuration's version, token/context
  limits, and engine-level optimization flags (e.g. FP8 quantization, tensor-parallel
  degree).

These CRDs are the same declarative pattern Kubernetes already uses for `Deployment` or
`Service` objects — the point is that AI-specific concerns (model, engine, quantization)
become first-class fields instead of being buried in a generic container spec.

### B. The Inference Gateway (IGW)

This is llm-d's "smart traffic cop," and it implements the upstream **Kubernetes Gateway
API Inference Extension** spec rather than being a bespoke llm-d-only mechanism.

Plain Kubernetes load-balances **round-robin**: if 5 replicas of a model are running, a
new request goes to whichever pod is next in rotation, with zero awareness of state.

The llm-d Inference Gateway instead does **prefix/KV-cache-aware routing**: it tracks
which pod already holds the KV cache for an ongoing multi-turn conversation (or for a
prompt sharing a long common prefix with a prior request) and routes follow-up requests
back to *that* pod. This avoids recomputing attention over text the cluster has already
processed, which directly cuts latency and GPU compute cost — this is functionally the
Kubernetes-native analogue of vLLM's own prefix caching, just applied across the whole
cluster instead of within a single engine instance.


## 4. Feature A — Prefill/Decode (P/D) Disaggregation <a id="4"></a>

Serving an LLM request has two phases with very different hardware profiles:

| Phase | What happens | Bottleneck |
|---|---|---|
| **Prefill** | The engine ingests the *entire* input prompt at once and computes the initial attention/KV-cache state | **Compute-bound** (parallel matrix math over the whole prompt) |
| **Decode** | The engine generates output tokens one at a time, each depending on the last | **Memory-bandwidth-bound** (repeatedly streaming model weights + KV cache from GPU memory) |

Running both phases on the same GPU (the default in a naive vLLM deployment) creates
**head-of-line blocking**: if a new request drops a huge document into prefill while
another user's session is mid-decode, the decode stream stalls — visible to that user as
sudden jitter/latency spikes mid-response.

**llm-d's fix:** physically separate the two phases onto different pods —

- **Prefill pods**: provisioned for raw compute throughput, sized to swallow large prompts fast.
- **Decode pods**: provisioned for memory bandwidth, sized to stream tokens smoothly.
- llm-d's controllers handle transferring the computed KV cache between the prefill and
  decode pod over the cluster's high-speed interconnect, so the split is transparent to
  the pipeline as a whole.

This is often referred to in the broader ecosystem as **"xPyD" (x prefill, y decode)**
disaggregated serving — you'll see that term in llm-d and related literature (e.g. vLLM's
own disaggregation work).


## 5. Feature B — Native Multi-Node Orchestration (LeaderWorkerSet) <a id="5"></a>

Large models (e.g. Llama-3-70B-class and up) don't fit in the memory of a single
consumer or even single-enterprise GPU (16–24 GB), and sometimes not even on a single
multi-GPU node. Serving them requires **tensor parallelism across multiple GPUs**,
potentially spanning multiple physical machines.

llm-d uses the **LeaderWorkerSet (LWS)** Kubernetes pattern to manage this:

- One pod is elected the **Leader**, responsible for the API-facing communication.
- The remaining pods are **Workers**, which the Leader coordinates for tensor-parallel /
  pipeline-parallel execution across GPUs and nodes.
- llm-d's orchestration handles the clustering setup natively, removing the need for
  hand-written multi-node launch scripts (the kind of glue code teams previously had to
  maintain per-model, per-cluster).

This is what makes llm-d relevant not just for small-model horizontal scaling, but for
serving frontier-scale open models in production.


## 6. How llm-d Fits the Broader Ecosystem <a id="6"></a>

llm-d is **not** a ground-up replacement for the inference engines or serving frameworks
you already use — it's a Kubernetes-native orchestration layer that composes with them:

- **vLLM** — the reference inference engine most `InferencePool`s run under the hood;
  llm-d's cluster-wide cache-aware routing is the multi-node extension of ideas vLLM
  already does inside a single process (prefix caching, continuous batching).
- **Kubernetes Gateway API Inference Extension** — the upstream, vendor-neutral spec that
  llm-d's Inference Gateway implements, meaning the routing behavior isn't a proprietary
  llm-d-only mechanism.
- **KServe / Kyverno / KAR** — llm-d is designed to be *composable* with adjacent CNCF
  projects: KServe for general model-serving lifecycle, Kyverno for policy governance on
  AI workloads, KAR for AI cluster conformance testing. llm-d specifically owns the
  distributed-inference-serving piece of that stack.
- **TGI (Hugging Face's Text Generation Inference)** — noted here as context: TGI was
  **archived in March 2026**, which is part of why vLLM has become the default reference
  engine across the ecosystem (including for llm-d), and is worth flagging as an update to
  your earlier [[ai-engineering-knowledge-base]] notebooks that listed TGI as a current
  peer option.


## 7. Comparison: llm-d vs. Other Serving Layers <a id="7"></a>

| Layer | What it actually is | Cluster-aware KV routing? | P/D disaggregation? | Multi-node tensor parallel? | Best for |
|---|---|---|---|---|---|
| **Plain vLLM (single instance)** | Inference engine/server | Within-instance only | Manual/experimental | Manual | Single-node serving, dev/test |
| **Ollama / llama.cpp** | Local/single-machine serving | No | No | No | Local dev, small-scale self-hosting |
| **llm-d** | Kubernetes orchestration layer *for* inference | **Yes**, cluster-wide | **Yes**, native | **Yes**, via LWS | Production, multi-tenant, multi-model K8s clusters |
| **KServe** | General ML model-serving CRD/lifecycle manager | No (routing-agnostic) | No | No (relies on the engine) | Standardizing model deploy/rollout across ML frameworks generally, not LLM-specific |
| **Ray Serve** | General-purpose distributed serving on Ray | No (application-level, not K8s-native) | Possible to build manually | Possible to build manually | Python-native distributed serving outside K8s-first shops |
| **Triton Inference Server** | Multi-framework model server | No | No | Limited | Classic ML + some LLM serving, strong non-LLM model support |

**Takeaway:** llm-d doesn't compete with vLLM — it's the Kubernetes control plane that
makes a *fleet* of vLLM (or TGI/Triton) instances behave like one cache-aware, phase-aware,
multi-node-capable inference cluster instead of a set of independent, dumb replicas.


## 8. Cheat Sheet & Key Terms <a id="8"></a>

| Term | Meaning |
|---|---|
| **IGW (Inference Gateway)** | The cache-aware, inference-state-aware traffic router; implements the K8s Gateway API Inference Extension |
| **InferencePool** | CRD declaring a pool of pods serving one model + engine combination |
| **InferenceModel** | CRD tracking a specific model's version, limits, and optimization flags |
| **Prefill** | Compute-bound phase: processing the full input prompt |
| **Decode** | Memory-bandwidth-bound phase: generating output tokens one at a time |
| **P/D Disaggregation (xPyD)** | Splitting prefill and decode across separate, differently-provisioned pods |
| **LeaderWorkerSet (LWS)** | K8s pattern for multi-node model serving with one Leader + N Worker pods |
| **KV cache** | Stored attention key/value tensors from prior tokens, reused to avoid recomputation |
| **Cache-aware routing** | Sending a request back to the pod that already holds its relevant KV cache |

### Quick decision guide
- Serving one small model on one box → **Ollama / llama.cpp / plain vLLM**, skip llm-d.
- Serving multiple models/tenants on a real Kubernetes cluster at scale, with GPUs as the
  bottleneck resource → **llm-d** is the layer built for exactly this.
- Model too large for one node → llm-d's **LeaderWorkerSet** support is a direct reason to
  adopt it rather than hand-rolling multi-node launch scripts.


## 9. Getting Started <a id="9"></a>

- Project site: **llm-d.ai**
- CNCF project page: **cncf.io/projects/llm-d**
- Source: `github.com/llm-d/llm-d`
- Typical entry point: install the Helm chart into an existing GPU-enabled Kubernetes
  cluster, then declare an `InferencePool` for your first model.

### Suggested next notebooks (natural follow-ons)
1. **An annotated `InferencePool` YAML manifest** — walk a real manifest field-by-field
   (model, engine, scale bounds, quantization flags) to connect this theory to a concrete
   deploy artifact.
2. **The Prefill/Decode network layer** — how KV cache actually moves between prefill and
   decode pods over the cluster interconnect, and what that implies for network topology
   choices (RDMA/InfiniBand vs. standard networking).
3. **llm-d vs. your existing [[mlops-p2]] deployment story** — since that pipeline already
   uses GCP + Docker, a worthwhile comparison would be GKE + llm-d vs. your current
   MLflow-based deployment approach, specifically for the serving layer.
